In [38]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from category_encoders import TargetEncoder
import joblib

In [39]:
df = df = pd.read_csv("../csv/clean_cars.csv")

df.head()

,brand,model,model_year,milage,fuel_type,transmission,ext_col,int_col,accident,clean_title,...,brand_Volvo,brand_smart,engine_liters,cylinders,turbo,horsepower,fuel_diesel,fuel_electric,fuel_gasoline,fuel_hybrid
0,Ford,Utility Police Interceptor Base,2013,51000,gasoline,6-Speed Automatic,Black,Black,1,1,...,False,False,3.7,6,0,300,0,0,1,0
1,Hyundai,Palisade SEL,2021,34742,gasoline,8-Speed Automatic,Moonlight Cloud,Gray,1,1,...,False,False,3.8,6,0,-1,0,0,1,0
2,Lexus,RX 350 RX 350,2022,22372,gasoline,Automatic,Blue,Black,0,0,...,False,False,3.5,-1,0,-1,0,0,1,0
3,INFINITI,Q50 Hybrid Sport,2015,88900,hybrid,7-Speed Automatic,Black,Black,0,1,...,False,False,3.5,6,0,354,0,0,0,1
4,Audi,Q3 45 S line Premium Plus,2021,9835,gasoline,8-Speed Automatic,White,Black,0,0,...,False,False,2.0,-1,1,-1,0,0,1,0


In [40]:
def evaluate_model(y_true, y_pred):
    return{
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred)
    }

In [41]:
# 1) Filtro de outliers
p99 = np.percentile(df["price"], 99)
df_filt = df[df["price"] <= p99].copy()

# 2) Nuevas features
df_filt["car_age"] = 2024 - df_filt["model_year"]
df_filt["milage_log"] = np.log1p(df_filt["milage"])
df_filt["age_x_milage"] = df_filt["car_age"] * df_filt["milage_log"]

# 3) One-hot de fuel_type y transmission
cat_cols = ["fuel_type", "transmission"]
df_filt_ohe = pd.get_dummies(df_filt, columns=cat_cols, drop_first=True)

# 4) Definir X e y finales
y = df_filt_ohe["price"]
X = df_filt_ohe.drop(columns=[
    "price",
    "brand",
    "int_col",
    "ext_col",
    "model"
])

y_log = np.log1p(y)

x_train, x_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

y_train_real = np.expm1(y_train)
y_test_real  = np.expm1(y_test)

In [52]:
# Random Forest
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=9,
    min_samples_leaf=9, # ⬇️ reduce ruido
    random_state=42,
    n_jobs=-1
)

rf.fit(x_train, y_train)

# Predicciones (log → real)
y_pred_rf_train = np.expm1(rf.predict(x_train))
y_pred_rf_test  = np.expm1(rf.predict(x_test))

In [53]:
importances = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importances.head(5)

age_x_milage     0.684501
milage_log       0.098263
milage           0.093117
horsepower       0.070030
engine_liters    0.025235
dtype: float64

In [54]:
y_pred_rf_train  = np.expm1(rf.predict(x_train))
y_pred_rf_test   = np.expm1(rf.predict(x_test))

mae_train_rf = mean_absolute_error(y_train_real, y_pred_rf_train)
mae_test_rf  = mean_absolute_error(y_test_real, y_pred_rf_test)

overfit_rf  = (mae_test_rf  - mae_train_rf)  / mae_train_rf  * 100

print("Overfitting (%) por modelo:")
print(f"RF:  {overfit_rf:.2f}%")

Overfitting (%) por modelo:
RF:  6.33%


In [55]:
# --- Asegurarnos de tener y_train_real ---
y_train_real = np.expm1(y_train)

# --- Evaluar modelos en TRAIN ---
results_train = {
    'RF': evaluate_model(y_train_real, y_pred_rf_train)
}

# --- Convertir a DataFrame para ver resultados ---
pd.DataFrame(results_train).T


,MAE,RMSE,R2
RF,12176.775321,23713.200086,0.506392


In [56]:
# Guardar modelos entrenados
joblib.dump(rf, "../models/rf_model.pkl")

# Guardar valores reales de test
joblib.dump(y_test_real, "../models/y_test_real.pkl")

# Guardar predicciones del test (en €)
joblib.dump(y_pred_rf_test,  "../models/y_pred_rf.pkl")

['../models/y_pred_rf.pkl']